In [ ]:
from tokenizers import Tokenizer

In [ ]:
# ============================================================
# DAY 30 — REAL-WORLD AI TEXT ANALYZER
# ============================================================
#
# WHAT THIS PROGRAM DOES:
#
#   User text
#       ↓
#   Hugging Face Tokenizer
#       ↓
#   Tokens
#       ↓
#   Token IDs
#       ↓
#   Token count
#       ↓
#   Pre-trained Sentiment Model
#       ↓
#   POSITIVE / NEGATIVE
#
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
import torch


# ============================================================
# 2. MODEL NAME
# ============================================================
#
# We are using a small pre-trained sentiment model.
#
# IMPORTANT:
#
# We are NOT training this model ourselves.
#
# Somebody already trained it.
#
# We are simply using it.
#
# ============================================================

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"


# ============================================================
# 3. START PROGRAM
# ============================================================

print("=" * 70)
print("              DAY 30 — AI TEXT ANALYZER")
print("=" * 70)

print()
print("Loading AI model...")
print("This may take some time the first time.")
print()


# ============================================================
# 4. LOAD TOKENIZER
# ============================================================
#
# The tokenizer's job:
#
#     TEXT
#       ↓
#     TOKENS
#       ↓
#     NUMBERS
#
# Example:
#
#     "I love pizza"
#
# might become something similar to:
#
#     ["i", "love", "pizza"]
#
# and then:
#
#     [1045, 2293, 10733]
#
# The exact IDs depend on the tokenizer.
#
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


# ============================================================
# 5. LOAD PRE-TRAINED AI MODEL
# ============================================================
#
# This is the actual neural network.
#
# It has already learned from a large dataset.
#
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
)


# ============================================================
# 6. PUT MODEL INTO EVALUATION MODE
# ============================================================
#
# We are using the model to make predictions.
#
# We are NOT training it.
#
# eval() tells PyTorch:
#
# "I'm only predicting now."
#
# ============================================================

model.eval()


# ============================================================
# 7. GET MODEL LABELS
# ============================================================
#
# The model has labels such as:
#
#     LABEL_0
#     LABEL_1
#
# We convert those into human-friendly names.
#
# ============================================================

id_to_label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}


# ============================================================
# 8. TOKENIZE FUNCTION
# ============================================================

def analyze_tokens(text):

    """
    Convert text into tokens and token IDs.
    """

    # --------------------------------------------------------
    # Encode the text
    # --------------------------------------------------------

    encoded = tokenizer(
        text,
        add_special_tokens=True,
        return_tensors="pt"
    )


    # --------------------------------------------------------
    # Get token IDs
    # --------------------------------------------------------

    token_ids = encoded["input_ids"][0]


    # --------------------------------------------------------
    # Convert IDs back into readable tokens
    # --------------------------------------------------------

    tokens = tokenizer.convert_ids_to_tokens(
        token_ids
    )


    return tokens, token_ids, encoded


# ============================================================
# 9. SENTIMENT FUNCTION
# ============================================================

def predict_sentiment(text):

    """
    Give the model some text and return
    its sentiment prediction.
    """

    # --------------------------------------------------------
    # Convert text into model input
    # --------------------------------------------------------

    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )


    # --------------------------------------------------------
    # Turn off gradient calculation
    # --------------------------------------------------------
    #
    # We aren't training.
    #
    # This saves memory and computation.
    #
    # --------------------------------------------------------

    with torch.no_grad():

        outputs = model(
            **encoded
        )


    # --------------------------------------------------------
    # Raw model scores
    # --------------------------------------------------------

    logits = outputs.logits


    # --------------------------------------------------------
    # Convert scores into probabilities
    # --------------------------------------------------------

    probabilities = torch.softmax(
        logits,
        dim=1
    )[0]


    # --------------------------------------------------------
    # Find the highest probability
    # --------------------------------------------------------

    predicted_id = torch.argmax(
        probabilities
    ).item()


    # --------------------------------------------------------
    # Convert ID into label
    # --------------------------------------------------------

    label = id_to_label[
        predicted_id
    ]


    # --------------------------------------------------------
    # Get confidence
    # --------------------------------------------------------

    confidence = probabilities[
        predicted_id
    ].item()


    return label, confidence, probabilities


# ============================================================
# 10. DISPLAY TOKEN INFORMATION
# ============================================================

def show_token_information(text):

    """
    Print everything related to tokenization.
    """

    tokens, token_ids, encoded = analyze_tokens(
        text
    )


    print()
    print("-" * 70)
    print("TOKENIZATION")
    print("-" * 70)


    # --------------------------------------------------------
    # Original text
    # --------------------------------------------------------

    print()
    print("Original text:")
    print(text)


    # --------------------------------------------------------
    # Tokens
    # --------------------------------------------------------

    print()
    print("Tokens:")

    print(tokens)


    # --------------------------------------------------------
    # Token IDs
    # --------------------------------------------------------

    print()
    print("Token IDs:")

    print(
        token_ids.tolist()
    )


    # --------------------------------------------------------
    # Number of tokens
    # --------------------------------------------------------

    print()
    print(
        "Number of tokens:",
        len(tokens)
    )


    # --------------------------------------------------------
    # Decode IDs back into text
    # --------------------------------------------------------

    decoded = tokenizer.decode(
        token_ids,
        skip_special_tokens=True
    )


    print()
    print("Decoded text:")

    print(decoded)


# ============================================================
# 11. DISPLAY SENTIMENT
# ============================================================

def show_sentiment(text):

    """
    Print sentiment prediction.
    """

    label, confidence, probabilities = (
        predict_sentiment(text)
    )


    print()
    print("-" * 70)
    print("SENTIMENT ANALYSIS")
    print("-" * 70)


    print()
    print("Prediction:")
    print(label)


    print()
    print("Confidence:")

    print(
        f"{confidence * 100:.2f}%"
    )


    print()
    print("Negative probability:")

    print(
        f"{probabilities[0].item() * 100:.2f}%"
    )


    print()
    print("Positive probability:")

    print(
        f"{probabilities[1].item() * 100:.2f}%"
    )


# ============================================================
# 12. COMPLETE ANALYSIS
# ============================================================

def analyze_text(text):

    """
    Run the complete Day 30 analysis.
    """

    print()
    print("=" * 70)
    print("                    ANALYSIS")
    print("=" * 70)


    # --------------------------------------------------------
    # Show tokenizer information
    # --------------------------------------------------------

    show_token_information(
        text
    )


    # --------------------------------------------------------
    # Show sentiment
    # --------------------------------------------------------

    show_sentiment(
        text
    )


# ============================================================
# 13. TEST SENTENCES
# ============================================================
#
# Before making the program interactive,
# let's test several sentences automatically.
#
# ============================================================

test_sentences = [

    "I absolutely loved this movie.",

    "This movie was terrible.",

    "The food was amazing and delicious.",

    "The service was horrible.",

    "I really enjoyed the experience.",

    "This was boring and disappointing.",

]


print()
print("=" * 70)
print("                  AUTOMATIC TESTS")
print("=" * 70)


for number, text in enumerate(
    test_sentences,
    start=1
):

    print()
    print(
        f"TEST {number}: {text}"
    )


    # --------------------------------------------------------
    # Token information
    # --------------------------------------------------------

    tokens, token_ids, _ = (
        analyze_tokens(text)
    )


    print(
        "Tokens:",
        tokens
    )


    print(
        "Token count:",
        len(tokens)
    )


    # --------------------------------------------------------
    # Sentiment
    # --------------------------------------------------------

    label, confidence, _ = (
        predict_sentiment(text)
    )


    print(
        "Sentiment:",
        label
    )


    print(
        f"Confidence: {confidence * 100:.2f}%"
    )


# ============================================================
# 14. INTERACTIVE MODE
# ============================================================

print()
print("=" * 70)
print("                  INTERACTIVE MODE")
print("=" * 70)

print()
print("Type any sentence and the AI will analyze it.")
print()
print("Commands:")
print("  quit  → stop the program")
print("  help  → show instructions")
print()


while True:

    # --------------------------------------------------------
    # Ask user for text
    # --------------------------------------------------------

    text = input(
        "\nEnter text: "
    )


    # --------------------------------------------------------
    # QUIT
    # --------------------------------------------------------

    if text.lower() == "quit":

        print()
        print("Goodbye!")

        break


    # --------------------------------------------------------
    # HELP
    # --------------------------------------------------------

    if text.lower() == "help":

        print()
        print("Example:")
        print(
            "I really enjoyed this movie!"
        )

        print()
        print(
            "The movie was terrible."
        )

        print()
        print(
            "Type 'quit' to exit."
        )

        continue


    # --------------------------------------------------------
    # EMPTY INPUT
    # --------------------------------------------------------

    if not text.strip():

        print(
            "Please enter some text."
        )

        continue


    # --------------------------------------------------------
    # ANALYZE
    # --------------------------------------------------------

    analyze_text(
        text
    )


# ============================================================
# 15. END
# ============================================================

print()
print("=" * 70)
print("                 DAY 30 COMPLETE")
print("=" * 70)